In [ ]:
import numpy as np
from sklearn.metrics import precision_recall_curve, f1_score, roc_auc_score, accuracy_score, auc,confusion_matrix,roc_curve
import pandas as pd
import matplotlib.pyplot as plt
import os

model_dir = os.getcwd()+ "/models/"
path = os.getcwd()+ "/data/EchoNext/"
model_file = model_dir + 'ECG_Echo_CNN/model.h5'

y_train = np.load(path + 'y_train.npy',allow_pickle=True)
y_test  = np.load(path + 'y_test.npy',allow_pickle=True)
y_pred_train = np.load(path + 'y_pred_train.npy',allow_pickle=True)
y_pred_test  = np.load(path + 'y_pred_test.npy',allow_pickle=True)

In [ ]:
df = pd.read_csv(path+"echonext_metadata_100k.csv")

df['y_true'] = np.nan
df['y_pred'] = np.nan

df.loc[df['split'] == 'train', 'y_true'] = y_train
df.loc[df['split'] == 'train', 'y_pred'] = y_pred_train
df.loc[df['split'] == 'test', 'y_true'] = y_test
df.loc[df['split'] == 'test', 'y_pred'] = y_pred_test
df = df[df['split'].isin(['train', 'test'])].copy()
df.to_csv(path+"echonext_metadata_train_test_ypred.csv", index=False)
df.columns

print(len(df))
df['split'].value_counts()

In [ ]:
def sklearn_metrics(y_true, y_pred):
    y_bin = np.copy(y_pred)
    y_bin[y_bin >= 0.5] = 1
    y_bin[y_bin < 0.5]  = 0

    # Compute area under precision-Recall curve
    precision, recall, thresholds = precision_recall_curve(y_true, y_pred)
    auc_sum = auc(recall, precision)

    TP = np.sum((y_true == 1) & (y_bin == 1))
    TN = np.sum((y_true == 0) & (y_bin == 0))
    FP = np.sum((y_true == 0) & (y_bin == 1))
    FN = np.sum((y_true == 1) & (y_bin == 0))

    sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
    print("Accuracy        : {:.2f}".format(accuracy_score(y_true.flatten(), y_bin.flatten())* 100))
    print("Macro AUC score : {:.2f}".format(roc_auc_score(y_true, y_pred, average='macro') * 100))
    print('AUPRC           : {:.2f}'.format(auc_sum  * 100))
    print("Macro F1 score  : {:.2f}".format(f1_score(y_true, y_bin, average='macro') * 100))
    print("Sensitivity     : {:.2f}".format(sensitivity * 100))
    print("Specificity     : {:.2f}".format(specificity * 100))

In [ ]:
print("Train")
sklearn_metrics(y_train.ravel(), y_pred_train.ravel())
print("\nTest")
sklearn_metrics(y_test.ravel(), y_pred_test.ravel())

y_all = np.concatenate([y_train, y_test], axis=0)
y_pred_all = np.concatenate([y_pred_train, y_pred_test], axis=0)
print("\nAll")
sklearn_metrics(y_all.ravel(), y_pred_all.ravel())

In [ ]:
cm = confusion_matrix(y_all.ravel(),(y_pred_all.ravel() >= 0.5).astype(int))
print(cm)

In [ ]:

# Compute precision-recall points
precision, recall, thresholds = precision_recall_curve(y_all, y_pred_all)

# Compute area under the PR curve
auprc = auc(recall, precision)

print("AUPRC:", auprc)

plt.plot(recall, precision, label=f"AUC = {auprc:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curve")
plt.legend(loc="upper right")
plt.grid()
plt.show()


In [ ]:

fpr, tpr, thresholds = roc_curve(y_all, y_pred_all)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], 'k--')  # diagonal for random classifier
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.grid()
plt.show()


In [ ]:
def eval_at_threshold(y_true, p, threshold):
    y_true = np.asarray(y_true).astype(int).ravel()
    p = np.asarray(p).ravel()
    y_hat = (p >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_hat).ravel()
    sens = tp / (tp + fn + 1e-12)
    spec = tn / (tn + fp + 1e-12)
    ppv  = tp / (tp + fp + 1e-12)
    npv  = tn / (tn + fn + 1e-12)
    return {
        "threshold": float(threshold),
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
        "sensitivity": float(sens),
        "specificity": float(spec),
        "PPV": float(ppv),
        "NPV": float(npv),
    }

In [ ]:
t= 0.11870113015174866
val_metrics = eval_at_threshold(y_test, y_pred_test, t)
print(val_metrics)

In [ ]:
#subgroup analyses
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_curve, f1_score, roc_auc_score, accuracy_score, auc

def _to_1d(a, name="array"):
    a = np.asarray(a)
    if a.ndim == 2 and a.shape[1] == 1:
        a = a[:, 0]
    if a.ndim != 1:
        raise ValueError(f"{name} must be 1D (or Nx1). Got shape={a.shape}")
    return a

def _metrics_at_threshold(y_true, y_prob, thr=0.5):
    y_true = _to_1d(y_true, "y_true").astype(int)
    y_prob = _to_1d(y_prob, "y_prob").astype(float)

    y_hat = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_hat, labels=[0, 1]).ravel()

    sens = tp / (tp + fn) if (tp + fn) else np.nan
    spec = tn / (tn + fp) if (tn + fp) else np.nan
    ppv  = tp / (tp + fp) if (tp + fp) else np.nan
    npv  = tn / (tn + fn) if (tn + fn) else np.nan
    acc  = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else np.nan
    pos_rate = (tp + fp) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else np.nan
    f1 = f1_score(y_true, y_hat, average='macro')
    # AUC requires both classes present
    auc_roc = np.nan
    if len(np.unique(y_true)) == 2:
        auc_roc = roc_auc_score(y_true, y_prob)

    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    auc_prc = auc(recall, precision) 

    return {
        "N": int(len(y_true)),
        "Pos": int(y_true.sum()),
        "Neg": int((1 - y_true).sum()),
        "AUC": auc_roc,
        "Threshold": thr,
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
        "Sensitivity": sens,
        "Specificity": spec,
        "PPV": ppv,
        "NPV": npv,
        "Accuracy": acc,
        "Macro F1 score": f1,
        "AUPRC" : auc_prc
    }

def _safe_value_counts(s):
    vc = s.value_counts(dropna=False)
    # keep it compact
    return {str(k): int(v) for k, v in vc.items()}

# ----------------------------
# build analysis DF + subgroup performance
# ----------------------------
def subgroup_performance(
    y_true,
    y_prob,
    meta_df,                  # must contain columns: age, sex, race 
    id_col=None,              # optional
    age_col="age",
    sex_col="sex",
    race_col="race",
    threshold=0.5,
    race_map=None,            # optional dict mapping raw race strings -> harmonized categories
    drop_unknown_race=False,  # optionally drop Unknown/NaN race
):
    df = meta_df.copy()

    y_true = _to_1d(y_true, "y_true")
    y_prob = _to_1d(y_prob, "y_prob")
    if len(y_true) != len(y_prob) or len(y_true) != len(meta_df):
        raise ValueError("Length mismatch among y_true, y_prob, and meta_df rows.")

    df = meta_df.copy()

    # attach y / prediction
    df["y_true"] = y_true.astype(int)
    df["y_prob"] = y_prob.astype(float)
    df["All"] = 1

    # Clean age
    df[age_col] = pd.to_numeric(df[age_col], errors="coerce")

    # Age bins
    age_bins = [-np.inf, 40, 60, 80, np.inf]
    age_labels = ["<40", "40-60", "60-80", ">80"]
    df["age_group"] = pd.cut(df[age_col], bins=age_bins, labels=age_labels, right=False)

    # Clean sex
    df[sex_col] = df[sex_col].astype("string").str.strip()
    sex_norm = {
        "m": "Male", "male": "Male",
        "f": "Female", "female": "Female",
    }
    df["sex_group"] = (
        df[sex_col].str.lower()
        .map(sex_norm)
        .fillna(df[sex_col])
        .fillna("Unknown")
    )

    # Clean race/ethnicity
    df[race_col] = df[race_col].astype("string").str.strip()
    if race_map is not None:
        df["race_group"] = df[race_col].map(race_map).fillna(df[race_col])
    else:
        df["race_group"] = df[race_col]
    df["race_group"] = df["race_group"].fillna("Unknown")

    if drop_unknown_race:
        df = df.loc[~df["race_group"].isin(["Unknown", "UNK", "NA", "N/A"])].copy()

    print("Age group counts:", _safe_value_counts(df["age_group"]))
    print("Sex counts:", _safe_value_counts(df["sex_group"]))
    print("Race counts:", _safe_value_counts(df["race_group"]))

    # Subgroup evaluation function
    def eval_by(group_col):
        rows = []
        for g, dfg in df.groupby(group_col, dropna=False):
            gname = str(g)
            m = _metrics_at_threshold(dfg["y_true"].values, dfg["y_prob"].values, thr=threshold)
            m.update({"GroupType": group_col, "Group": gname})
            rows.append(m)
        return pd.DataFrame(rows)

    perf = pd.concat(
        [eval_by("All"), eval_by("split"), eval_by("age_group"), eval_by("sex_group"), eval_by("race_group")],
        ignore_index=True,
    )

    # Order columns 
    col_order = [
        "GroupType", "Group", "N", "Pos", "Neg", "Threshold", "AUC", 
        "Sensitivity", "Specificity", "PPV", "NPV", "Accuracy", "Macro F1 score","AUPRC",
        "TP", "FP", "TN", "FN"
    ]
    perf = perf[col_order].sort_values(["GroupType", "Group"]).reset_index(drop=True)

    return df, perf


In [ ]:
meta=df
meta.columns

In [ ]:

df_combined, perf_table = subgroup_performance(
     y_true=meta["y_true"],
     y_prob=meta["y_pred"],
     meta_df=meta,
     age_col="age_at_ecg",
     sex_col="sex",
     race_col="race_ethnicity",
     threshold=0.5,  # use your chosen threshold if desired
     race_map={         
         "white": "White",
         "black": "Black",
         "Asian": "Asian",
         "Hispanic or Latino": "Hispanic",
         "Unknown": "Unknown",
     }
 )

print(perf_table)
perf_table.to_csv("subgroup_performance.csv", index=False)


In [ ]:
df_combined, perf_table = subgroup_performance(
     y_true=meta["y_true"],
     y_prob=meta["y_pred"],
     meta_df=meta,
     age_col="age_at_ecg",
     sex_col="sex",
     race_col="race_ethnicity",
     threshold= 0.11870113015174866,  # use chosen threshold 
     race_map={
         
         "white": "White",
         "black": "Black",
         "Asian": "Asian",
         "Hispanic or Latino": "Hispanic",
         "Unknown": "Unknown",
     }
 )

print(perf_table)
perf_table.to_csv("subgroup_performance_thr.csv", index=False)